# 08 Cyclic-permutation re-evaluation, story-disjoint split

Applies the cyclic permutation of notebook 06 to the five story-disjoint adapters. No training.

- Adapters: seed 42 `tier4_v2_fixed/adapter_groupsplit` (SHA-256 prefix `029b209a`), seeds 43-46
  from notebook 05. Loaded with unsloth, as in notebook 06.
- ToMBench: the 837-item story-disjoint test split, taken from the split files in the order of
  `groupsplit_base_ToMBench.csv` (prompt hashes checked). External benchmarks: the same manifests as
  notebook 06.
- All rotations, including shift 0, are run for every adapter in one environment; shift 0 is only
  compared with the existing CSVs.
- Base model: the external benchmarks are taken from notebook 06; only the 837 ToMBench items are
  run here.

Inferences per adapter: 17,379, about 1.9 hours on an L4.

Run the cells in order: [0-1] installs the pinned package versions; restart the runtime
(Runtime > Restart session) and continue from [0-2]. Results are written to Google Drive, and
cells that find their output already saved skip it, so an interrupted run can be resumed by
running [0-2] again and then the interrupted cell.

`DRIVE_DIR` is the Drive folder used by `notebooks/tombench_slm_qlora_complete_pipeline.ipynb`;
`tier4_v2_fixed/` inside it holds that notebook's outputs (split and manifest files, per-item
CSVs, adapters). The other folders are created by the revision notebooks; see
`notebooks/revision/README.md` for where their files are in `results/`.

## [0] Setup

In [ ]:
# -- [0-1] Install packages (versions pinned to requirements.lock.txt) --
import time, torch
if not torch.cuda.is_available():
    print('\nNo GPU detected. Runtime -> Change runtime type -> select a GPU.'); raise SystemExit
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB VRAM)')
t0 = time.time()
get_ipython().system('pip install -q unsloth==2026.8.18 unsloth_zoo==2026.8.12')
get_ipython().system('pip install -q transformers==5.5.0 trl==0.24.0 accelerate==1.14.0 bitsandbytes==0.50.1 peft==0.20.0')
get_ipython().system('pip install -q "datasets==3.6.0" tqdm')
print(f'   {time.time()-t0:.0f}s  ->  restart the runtime, then run from [0-2]')

In [ ]:
# -- [0-2] Mount Drive, set paths, helpers, manifests --
from google.colab import drive
drive.mount('/content/drive')

import os, re, json as _json, time, hashlib, platform, string
import pandas as pd, numpy as np, torch
import transformers, peft, unsloth
from unsloth import FastLanguageModel
transformers.logging.set_verbosity_error()
assert transformers.__version__ == '5.5.0', f'transformers version not pinned: {transformers.__version__}'

DRIVE_DIR   = '/content/drive/MyDrive/ToMBench_Results'
RESULTS_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
SD_DIR      = f'{DRIVE_DIR}/storydisjoint_5seed'
P2_DIR      = f'{DRIVE_DIR}/cyclic_itemlevel'
OUT_DIR     = f'{DRIVE_DIR}/cyclic_storydisjoint'
os.makedirs(OUT_DIR, exist_ok=True)

BENCHMARKS = ['ToMBench', 'ToMi', 'OpenToM', 'SocialIQa', 'HiToM']
EXTERNAL = ['ToMi', 'OpenToM', 'SocialIQa', 'HiToM']
SEEDS = [42, 43, 44, 45, 46]
MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048
LETTERS = list(string.ascii_uppercase)

def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()[:16]

ADAPTERS = {42: f'{RESULTS_DIR}/adapter_groupsplit'}
ADAPTERS.update({s: f'{SD_DIR}/adapter_groupsplit_seed{s}' for s in [43, 44, 45, 46]})
ADAPTER_HASH = {}
for s, p in ADAPTERS.items():
    f = f'{p}/adapter_model.safetensors'
    assert os.path.exists(f), f'seed{s} story-disjoint adapter missing: {f}'
    ADAPTER_HASH[s] = sha16(f)
    print(f'seed{s}: {p}  sha256[:16]={ADAPTER_HASH[s]}')
assert ADAPTER_HASH[42] == '029b209a2f1cfc5b', 'seed-42 story-disjoint adapter is not the retained 2026-09-03 run -- stop and check'
assert len(set(ADAPTER_HASH.values())) == 5, 'the same adapter appears twice'

SYSTEM_PROMPT = (
    'Below is a multiple-choice question with a story and several answer options. '
    'Based on the content of the story and the given question, please infer the most likely answer '
    'and output the answer index in the format [[X]] where X is one of A, B, C, D, E.'
)
ANS_RE = re.compile(r'\[\[\s*([A-E])\s*\]\]', re.IGNORECASE)
FB_RE = re.compile(r'\b([A-E])\b')

def extract_letter(text):
    m = ANS_RE.search(text)
    if m: return m.group(1).upper()
    m = FB_RE.search(text)
    if m: return m.group(1).upper()
    return 'A'

def to_text(tokenizer, user_prompt):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_prompt}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def build_user_prompt(story, question, options):
    lines = [f'{l}. {t}' for l, t in options]
    return f'[Story]\n{story}\n\n[Question]\n{question}\n\n[Candidate Answers]\n' + '\n'.join(lines)

def parse_options(user_prompt):
    m = re.search(r'\[Candidate Answers\]\n(.+)', user_prompt, re.S)
    block = m.group(1) if m else user_prompt
    return [(l, t.strip()) for l, t in re.findall(r'^([A-E])\.\s*(.+?)$', block, re.M)]

def prep(items):
    for it in items:
        it['options'] = parse_options(it['user_prompt'])
        n = len(it['options'])
        assert [l for l, _ in it['options']] == LETTERS[:n] and it['answer'] in LETTERS[:n], it['item_id']
        assert build_user_prompt(it['story'], it['question'], it['options']) == it['user_prompt'], it['item_id']
        it['n_options'] = n
    return items

def load_storydisjoint_test():
    # take the 837 story-disjoint test items, in the order of the released CSV, from all 2,860 items (the three item-level split files)
    allit = {}
    for f in ['split_train_full_v2.json', 'split_val_full_v2.json', 'split_test_full_v2.json']:
        with open(f'{RESULTS_DIR}/{f}', encoding='utf-8') as fh:
            for it in _json.load(fh): allit[it['item_id']] = it
    assert len(allit) == 2860
    ref = pd.read_csv(f'{RESULTS_DIR}/groupsplit_base_ToMBench.csv')
    items = [dict(allit[i]) for i in ref['item_id']]
    assert len(items) == 837
    assert all(it['prompt_hash'] == ph for it, ph in zip(items, ref['prompt_hash'])), 'prompt_hash mismatch'
    return items

def load_external(bm):
    with open(f'{RESULTS_DIR}/manifest_{bm}.json', encoding='utf-8') as f:
        return _json.load(f)

MANIFESTS = {'ToMBench': prep(load_storydisjoint_test())}
for bm in EXTERNAL: MANIFESTS[bm] = prep(load_external(bm))
for bm in BENCHMARKS:
    vc = pd.Series([it['n_options'] for it in MANIFESTS[bm]]).value_counts().sort_index().to_dict()
    print(bm, len(MANIFESTS[bm]), vc, 'rotations', sum(it['n_options'] for it in MANIFESTS[bm]))

def cyclic_shift(options, k):
    n = len(options)
    texts = [t for _, t in options]
    shifted = texts[-k % n:] + texts[:-k % n] if k % n else texts
    new_options = [(LETTERS[i], shifted[i]) for i in range(n)]
    remap = {LETTERS[i]: LETTERS[(i + k) % n] for i in range(n)}
    return new_options, remap

for it in MANIFESTS['HiToM'][:20] + MANIFESTS['OpenToM'][:20] + MANIFESTS['ToMBench'][:50]:
    gold_text = dict(it['options'])[it['answer']]
    for k in range(it['n_options']):
        opts, remap = cyclic_shift(it['options'], k)
        assert dict(opts)[remap[it['answer']]] == gold_text
print('cyclic_shift self-check passed')

@torch.no_grad()
def infer_one(model, tokenizer, user_prompt, letter_ids, lid):
    inputs = tokenizer(to_text(tokenizer, user_prompt), return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=12, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id,
                         output_scores=True, return_dict_in_generate=True)
    gen_ids = out.sequences[0][inputs['input_ids'].shape[1]:]
    gen = tokenizer.decode(gen_ids, skip_special_tokens=True)
    margin = None
    for step, tok in enumerate(gen_ids.tolist()):
        if tok in letter_ids and step < len(out.scores):
            sc = out.scores[step][0].float().cpu()[lid]
            v, _ = torch.sort(sc, descending=True)
            margin = float(v[0] - v[1])
            break
    return extract_letter(gen), gen, margin

def letter_token_ids(tokenizer):
    ids = {}
    for L in 'ABCDE':
        t = tokenizer.encode(L, add_special_tokens=False)
        assert len(t) == 1, f'{L} is not a single token: {t}'
        ids[t[0]] = L
    return ids, torch.tensor(list(ids.keys()))

def out_csv(model_tag, bm):
    return f'{OUT_DIR}/cyclic_sd_{model_tag}_{bm}.csv'

def n_rot(bm):
    return sum(it['n_options'] for it in MANIFESTS[bm])

def run_cyclic(model_tag, model, tokenizer, benchmarks):
    FastLanguageModel.for_inference(model)
    letter_ids, lid = letter_token_ids(tokenizer)
    from tqdm.auto import tqdm
    for bm in benchmarks:
        path = out_csv(model_tag, bm)
        rows = pd.read_csv(path).to_dict('records') if os.path.exists(path) else []
        done = {(r['item_id'], int(r['shift_k'])) for r in rows}
        todo = [(it, k) for it in MANIFESTS[bm] for k in range(it['n_options']) if (it['item_id'], k) not in done]
        if not todo:
            print(f'[skip] {model_tag} {bm}: done ({len(rows)} rows)'); continue
        print(f'{model_tag} {bm}: {len(done)} rows done, {len(todo)} to go')
        t0 = time.time()
        for j, (it, k) in enumerate(tqdm(todo, desc=f'{model_tag}-{bm}')):
            opts, remap = cyclic_shift(it['options'], k)
            inv = {v: o for o, v in remap.items()}
            prompt = build_user_prompt(it['story'], it['question'], opts)
            pred, raw, margin = infer_one(model, tokenizer, prompt, letter_ids, lid)
            new_gold = remap[it['answer']]
            rows.append({'model': model_tag, 'benchmark': bm, 'item_id': it['item_id'], 'task': it.get('task'),
                         'n_options': it['n_options'], 'shift_k': k, 'gold_orig': it['answer'], 'new_gold': new_gold,
                         'pred': pred, 'pred_orig_letter': inv.get(pred), 'correct': int(pred == new_gold),
                         'letter_margin': margin, 'raw': raw})
            if (j + 1) % 500 == 0:
                pd.DataFrame(rows).to_csv(path, index=False)
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f'[saved] {model_tag} {bm}: {len(rows)} rows ({(time.time()-t0)/60:.1f} min)')

meta = {'gpu': torch.cuda.get_device_name(0), 'cuda': torch.version.cuda, 'torch': torch.__version__,
        'transformers': transformers.__version__, 'peft': peft.__version__, 'unsloth': unsloth.__version__,
        'python': platform.python_version(), 'adapter_sha16': ADAPTER_HASH,
        'started': time.strftime('%Y-%m-%d %H:%M:%S')}
with open(f'{OUT_DIR}/session_meta_{time.strftime("%Y%m%d_%H%M")}.json', 'w') as f:
    _json.dump(meta, f, indent=2)
print(meta)

---
## [1] Cyclic-permutation inference (base ToMBench, then seeds 42-46)

In [ ]:
# -- [1] Cyclic-permutation loop --
# base: the four external benchmarks come from notebook 06, so only the 837 story-disjoint ToMBench items are run
if not (os.path.exists(out_csv('base', 'ToMBench')) and len(pd.read_csv(out_csv('base', 'ToMBench'))) == n_rot('ToMBench')):
    base_model, base_tok = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    run_cyclic('base', base_model, base_tok, ['ToMBench'])
    del base_model, base_tok; torch.cuda.empty_cache()
else:
    print('[skip] base ToMBench done')
for bm in EXTERNAL:
    assert os.path.exists(f'{P2_DIR}/cyclic_base_{bm}.csv'), f'base rotations from notebook 06 missing: {bm}'

for s in SEEDS:
    tag = f'sdseed{s}'
    have = sum(len(pd.read_csv(out_csv(tag, bm))) for bm in BENCHMARKS if os.path.exists(out_csv(tag, bm)))
    need = sum(n_rot(bm) for bm in BENCHMARKS)
    if have >= need:
        print(f'[skip] {tag} done'); continue
    model, tok = FastLanguageModel.from_pretrained(model_name=ADAPTERS[s], max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    run_cyclic(tag, model, tok, BENCHMARKS)
    del model, tok; torch.cuda.empty_cache()
print('\n[1] all done')

---
## [2] Summary (no GPU needed)

- completeness of the rotations, and shift 0 against the existing CSVs
- `acc_orig`, `lenient`, `strict`, `consistent` as in notebook 06
- five-seed change, between-seed 95% t-interval, and the rule used in the paper: a direction is
  stated only when all three of lenient, strict and consistent exclude 0 with the same sign
- by option count, by OpenToM subtask, chosen-letter distribution; comparison with notebook 06

`code/analyze_cyclic.py` recomputes these tables from the files released in `results/`.

In [ ]:
# -- [2-1] Collect and check --
from google.colab import drive
drive.mount('/content/drive')
import os, pandas as pd, numpy as np
from scipy.stats import t as _t
pd.set_option('display.width', 220)
DRIVE_DIR   = '/content/drive/MyDrive/ToMBench_Results'
RESULTS_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
SD_DIR      = f'{DRIVE_DIR}/storydisjoint_5seed'
P2_DIR      = f'{DRIVE_DIR}/cyclic_itemlevel'
OUT_DIR     = f'{DRIVE_DIR}/cyclic_storydisjoint'
BENCHMARKS = ['ToMBench', 'ToMi', 'OpenToM', 'SocialIQa', 'HiToM']
EXTERNAL = ['ToMi', 'OpenToM', 'SocialIQa', 'HiToM']
SEEDS = [42, 43, 44, 45, 46]
MODELS = ['base'] + [f'sdseed{s}' for s in SEEDS]

frames = [pd.read_csv(f'{OUT_DIR}/cyclic_sd_base_ToMBench.csv')]
for bm in EXTERNAL:
    d = pd.read_csv(f'{P2_DIR}/cyclic_base_{bm}.csv'); d['model'] = 'base'; frames.append(d)
for s in SEEDS:
    for bm in BENCHMARKS:
        p = f'{OUT_DIR}/cyclic_sd_sdseed{s}_{bm}.csv'
        if os.path.exists(p): frames.append(pd.read_csv(p))
        else: print(f'[missing] sdseed{s} {bm}')
allr = pd.concat(frames, ignore_index=True)

cnt = allr.groupby(['model', 'benchmark', 'item_id']).agg(n=('shift_k', 'nunique'), K=('n_options', 'first'))
print('items with missing rotations:', int((cnt['n'] != cnt['K']).sum()))
print(cnt.reset_index().groupby(['model', 'benchmark']).size().unstack())

def ref_path(tag, bm):
    if tag == 'base': return f'{RESULTS_DIR}/groupsplit_base_ToMBench.csv' if bm == 'ToMBench' else f'{RESULTS_DIR}/baseline_{bm}.csv'
    s = int(tag.replace('sdseed', ''))
    return f'{RESULTS_DIR}/groupsplit_finetuned_{bm}.csv' if s == 42 else f'{SD_DIR}/groupsplit_finetuned_seed{s}_{bm}.csv'
print('\n[shift 0 vs existing result CSVs] mismatching items')
for tag in MODELS:
    for bm in BENCHMARKS:
        a = allr[(allr['model'] == tag) & (allr['benchmark'] == bm) & (allr['shift_k'] == 0)]
        if not len(a) or not os.path.exists(ref_path(tag, bm)): continue
        r = pd.read_csv(ref_path(tag, bm))
        mg = a.merge(r[['item_id', 'pred']], on='item_id', suffixes=('', '_ref'))
        print(f'  {tag:8s} {bm:9s} {int((mg["pred"] != mg["pred_ref"]).sum()):4d}/{len(mg)}')
allr.to_csv(f'{OUT_DIR}/cyclic_sd_all_long.csv', index=False)

In [ ]:
# -- [2-2] Measures, five-seed change, decision rule --
M = ['acc_orig', 'lenient', 'strict', 'consistent']
def item_metrics(g):
    return pd.Series({'acc_orig': g.loc[g['shift_k'] == 0, 'correct'].mean(), 'lenient': g['correct'].mean(),
                      'strict': float(g['correct'].min() == 1),
                      'consistent': float(g['pred_orig_letter'].nunique() == 1)})
per_item = allr.groupby(['model', 'benchmark', 'item_id', 'n_options']).apply(item_metrics).reset_index()
per_item.to_csv(f'{OUT_DIR}/cyclic_sd_per_item.csv', index=False)

def summarize(df, keys):
    s = df.groupby(keys)[M].mean() * 100
    s['n_items'] = df.groupby(keys).size()
    return s
summ = summarize(per_item, ['model', 'benchmark'])
print('[model x benchmark (%)]')
print(summ.round(2).unstack('model').swaplevel(axis=1).sort_index(axis=1).to_string())

base = summ.xs('base', level='model')
delta = pd.concat([(summ.xs(f'sdseed{s}', level='model')[M] - base[M]).assign(seed=s).reset_index() for s in SEEDS])
delta.to_csv(f'{OUT_DIR}/cyclic_sd_delta_by_seed.csv', index=False)
agg = delta.groupby('benchmark')[M].agg(['mean', 'std', 'count'])
for m in M:
    n = agg[(m, 'count')]; half = _t.ppf(0.975, n - 1) * agg[(m, 'std')] / np.sqrt(n)
    agg[(m, 'ci_lo')] = agg[(m, 'mean')] - half; agg[(m, 'ci_hi')] = agg[(m, 'mean')] + half
    agg[(m, 'excl0')] = (agg[(m, 'ci_lo')] > 0) | (agg[(m, 'ci_hi')] < 0)
agg = agg.sort_index(axis=1).reindex(BENCHMARKS)
print('\n[story-disjoint tuned - base change, five-seed mean ± SD, 95% interval, pp]')
print(agg.round(2).to_string())
agg.to_csv(f'{OUT_DIR}/cyclic_sd_delta_5seed.csv')

# a direction is stated only when lenient, strict and consistent all exclude 0 with the same sign
rule = []
for bm in BENCHMARKS:
    signs = [np.sign(agg.loc[bm, (m, 'mean')]) for m in ['lenient', 'strict', 'consistent']]
    ex = [bool(agg.loc[bm, (m, 'excl0')]) for m in ['lenient', 'strict', 'consistent']]
    rule.append({'benchmark': bm, 'lenient_excl0': ex[0], 'strict_excl0': ex[1], 'consistent_excl0': ex[2],
                 'direction': ('+' if signs[0] > 0 else '−') if all(ex) and len(set(signs)) == 1 else 'none'})
print('\n[decision rule: all three measures exclude 0, same sign]')
print(pd.DataFrame(rule).to_string(index=False))

# comparison: item-level cyclic permutation (notebook 06), five seeds
p2 = pd.read_csv(f'{P2_DIR}/cyclic_delta_by_seed.csv')
cmp_rows = []
for bm in BENCHMARKS:
    a = p2[p2['benchmark'] == bm]['lenient']; b = delta[delta['benchmark'] == bm]['lenient']
    cmp_rows.append({'benchmark': bm, 'item_level_lenient': f'{a.mean():+.2f} ± {a.std():.2f}',
                     'story_disjoint_lenient': f'{b.mean():+.2f} ± {b.std():.2f}'})
print('\n[lenient change: item-level vs story-disjoint]')
print(pd.DataFrame(cmp_rows).to_string(index=False))

In [ ]:
# -- [2-3] By option count, OpenToM subtask, chosen-letter distribution --
summ_k = summarize(per_item, ['model', 'benchmark', 'n_options'])
bk = summ_k.xs('base', level='model')
dk = pd.concat([(summ_k.xs(f'sdseed{s}', level='model')[['acc_orig', 'lenient', 'strict']] - bk[['acc_orig', 'lenient', 'strict']]).assign(seed=s).reset_index() for s in SEEDS])
out = dk.groupby(['benchmark', 'n_options'])[['acc_orig', 'lenient', 'strict']].agg(['mean', 'std']).round(2)
out['n_items'] = bk['n_items']
print('[change by option count, five-seed mean ± SD, pp]'); print(out.to_string())
out.to_csv(f'{OUT_DIR}/cyclic_sd_delta_by_noptions.csv')

po = per_item[per_item['benchmark'] == 'OpenToM'].merge(
    allr[allr['benchmark'] == 'OpenToM'].drop_duplicates(['item_id'])[['item_id', 'task']], on='item_id')
print('\n[OpenToM accuracy by subtask (%)]')
print((po.groupby(['task', 'model'])[['acc_orig', 'lenient', 'strict']].mean() * 100).round(2).unstack('model').to_string())

rows = []
for (m, bm, K), g in allr.groupby(['model', 'benchmark', 'n_options']):
    dist = (g['pred'].value_counts(normalize=True) * 100).reindex(list('ABCDE')[:K], fill_value=0)
    rows.append({'model': m, 'benchmark': bm, 'K': K, 'max_minus_min': round(dist.max() - dist.min(), 1), **dist.round(1).to_dict()})
lp = pd.DataFrame(rows); lp.to_csv(f'{OUT_DIR}/cyclic_sd_letter_prior.csv', index=False)
print('\n[chosen-letter distribution (%), all rotations] uniform = 100/K'); print(lp.to_string(index=False))